# Olist Data Quality Check

## 목적

분석에 앞서 데이터의 품질을 검증한다.

결측치, 중복 데이터, Primary Key의 유일성,
데이터 범위 등을 확인하여 분석에 적합한지 검토한다.

In [2]:
from pathlib import Path
import sqlite3
import pandas as pd

BASE_DIR = Path.cwd().parent
DB_PATH = BASE_DIR / "database" / "olist_dashboard.db"

conn = sqlite3.connect(DB_PATH)

## 1. Row Count

각 테이블의 데이터 건수를 확인하여 데이터가 정상적으로 적재되었는지 검증한다.

In [4]:
tables = [
    "customers",
    "geolocation",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "products",
    "product_category_name_translation",
    "sellers"
]

for table in tables:
    query = f"SELECT COUNT(*) AS row_count FROM {table}"
    result = pd.read_sql(query, conn)

    print(f"{table:<35} {result.loc[0, 'row_count']:,}")

customers                           99,441
geolocation                         1,000,163
order_items                         112,650
order_payments                      103,886
order_reviews                       99,224
orders                              99,441
products                            32,951
product_category_name_translation   71
sellers                             3,095


### 결과

- 모든 테이블이 정상적으로 SQLite 데이터베이스에 적재되었다.
- 원본 CSV와 동일한 행 수를 유지하고 있어 데이터 손실은 확인되지 않았다.

## 2. Missing Values

각 테이블의 결측치를 확인하여 분석에 영향을 줄 수 있는 컬럼이 있는지 검토한다.

In [7]:
for table in tables:
    df = pd.read_sql(f"SELECT * FROM {table}", conn)
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(f"\n[{table}]")
        print(missing)


[order_reviews]
review_comment_title      87656
review_comment_message    58247
dtype: int64

[orders]
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

[products]
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


### 결과

- 결측치는 `orders`, `products`, `order_reviews` 테이블에서만 확인되었다.
- 대부분 배송 정보 또는 리뷰 내용과 같이 업무 특성상 비어 있을 수 있는 컬럼이었다.
- 분석에 필요한 핵심 식별자(Primary Key)에서는 결측치가 발견되지 않았다.

## 3. Duplicate Check

Primary Key가 중복 없이 고유하게 존재하는지 확인한다.

Composite Key를 사용하는 테이블은 두 개 이상의 컬럼을 함께 검증한다.

In [10]:
pk_dict = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "product_category_name_translation": ["product_category_name"]
}

In [11]:
for table, pk_columns in pk_dict.items():
    df = pd.read_sql(f"SELECT * FROM {table}", conn)
    duplicate_count = df.duplicated(subset=pk_columns).sum()
    print(f"{table:<35} {duplicate_count:,}")

customers                           0
orders                              0
order_items                         0
order_payments                      0
order_reviews                       814
products                            0
sellers                             0
product_category_name_translation   0


### 추가 검증: Order Reviews

`order_reviews` 테이블에서 `review_id` 기준 중복이 814건 확인되었다.

따라서 `review_id`를 단독 식별자로 보기 전에 중복 데이터의 구조를 확인하고,
`review_id`와 `order_id` 조합의 고유성을 추가로 검증한다.

In [15]:
order_reviews = pd.read_sql("""
SELECT *
FROM order_reviews;
""", conn)

In [17]:
review_id_duplicates = order_reviews.duplicated(subset=["review_id"]).sum()

review_order_duplicates = order_reviews.duplicated(subset=["review_id", "order_id"]).sum()

print(f"review_id 기준 중복: {review_id_duplicates:,}")
print(f"review_id + order_id 기준 중복: {review_order_duplicates:,}")

review_id 기준 중복: 814
review_id + order_id 기준 중복: 0


### 확인 결과

- `review_id` 단독 기준으로는 814건의 중복이 존재한다.
- `review_id`와 `order_id` 조합은 중복 없이 고유하다.
- 따라서 데이터 품질 검증에서는 `review_id`와 `order_id` 조합을 기준으로 리뷰 데이터의 고유성을 확인하였다.

### 추가 검증 : Order Reviews

In [19]:
pk_dict = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id", "order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "product_category_name_translation": ["product_category_name"]
}

In [21]:
for table, pk_columns in pk_dict.items():
    df = pd.read_sql(f"SELECT * FROM {table}", conn)
    duplicate_count = df.duplicated(subset=pk_columns).sum()
    print(f"{table:<35} {duplicate_count:,}")

customers                           0
orders                              0
order_items                         0
order_payments                      0
order_reviews                       0
products                            0
sellers                             0
product_category_name_translation   0


### 결과

- 대부분의 테이블은 Primary Key 기준 중복이 존재하지 않았다.
- `order_reviews` 테이블에서는 `review_id` 단독 기준으로 814건의 중복이 확인되었다.
- 추가 검증 결과 `review_id`와 `order_id` 조합은 중복 없이 고유하였다.
- 따라서 데이터 품질 검증에서는 두 컬럼의 조합을 기준으로 리뷰 데이터의 고유성을 확인하였다.

## 4. Date Range

주문 및 배송 관련 날짜 컬럼의 최소·최대값을 확인하여 데이터의 분석 기간과 시간 범위를 검증한다.

In [23]:
date_range = pd.read_sql("""
SELECT
    MIN(order_purchase_timestamp) AS purchase_start,
    MAX(order_purchase_timestamp) AS purchase_end,
    MIN(order_delivered_customer_date) AS delivery_start,
    MAX(order_delivered_customer_date) AS delivery_end
FROM orders;
""", conn)

date_range

,purchase_start,purchase_end,delivery_start,delivery_end
0,2016-09-04 21:15:19,2018-10-17 17:30:18,2016-10-11 13:46:32,2018-10-17 13:22:46


In [25]:
date_columns = [
    "purchase_start",
    "purchase_end",
    "delivery_start",
    "delivery_end"
]

for column in date_columns:
    date_range[column] = pd.to_datetime(date_range[column])

analysis_days = (
    date_range.loc[0, "purchase_end"]
    - date_range.loc[0, "purchase_start"]
).days

print(date_range)
print(f"\nAnalysis Period: {analysis_days:,} days")

       purchase_start        purchase_end      delivery_start  \
0 2016-09-04 21:15:19 2018-10-17 17:30:18 2016-10-11 13:46:32   

         delivery_end  
0 2018-10-17 13:22:46  

Analysis Period: 772 days


In [27]:
date_range.loc[0, "purchase_end"]

Timestamp('2018-10-17 17:30:18')

### 결과

- 주문 데이터는 2016년 9월부터 2018년 10월까지의 기간을 포함한다.
- 전체 주문 데이터의 분석 기간은 772일이다.
- 배송 완료 기간도 함께 확인하여 주문일과 배송 완료일의 시간 범위를 검증하였다.
- 이후 월별 KPI 및 Tableau Dashboard 분석에서는 `order_purchase_timestamp`를 기본 날짜 기준으로 활용한다.

## 5. Foreign Key Integrity Check

Foreign Key가 참조하는 부모 테이블에 정상적으로 존재하는지 확인하여 테이블 간 참조 무결성을 검증한다.

In [35]:
fk_checks = {
    "orders → customers": {
        "child_table": "orders",
        "child_column": "customer_id",
        "parent_table": "customers",
        "parent_column": "customer_id"
    },
    "order_items → orders": {
        "child_table": "order_items",
        "child_column": "order_id",
        "parent_table": "orders",
        "parent_column": "order_id"
    },
    "order_items → products": {
        "child_table": "order_items",
        "child_column": "product_id",
        "parent_table": "products",
        "parent_column": "product_id"
    },
    "order_items → sellers": {
        "child_table": "order_items",
        "child_column": "seller_id",
        "parent_table": "sellers",
        "parent_column": "seller_id"
    },
    "order_payments → orders": {
        "child_table": "order_payments",
        "child_column": "order_id",
        "parent_table": "orders",
        "parent_column": "order_id"
    },
    "order_reviews → orders": {
        "child_table": "order_reviews",
        "child_column": "order_id",
        "parent_table": "orders",
        "parent_column": "order_id"
    }
}

In [37]:
for relationship, info in fk_checks.items():
    query = f"""
    SELECT COUNT(*) AS unmatched_count
    FROM {info["child_table"]} AS child
    LEFT JOIN {info["parent_table"]} AS parent
        ON child.{info["child_column"]} = parent.{info["parent_column"]}
    WHERE parent.{info["parent_column"]} IS NULL;
    """

    result = pd.read_sql(query, conn)
    unmatched_count = result.loc[0, "unmatched_count"]

    print(f"{relationship:<35} {unmatched_count:,}")

orders → customers                  0
order_items → orders                0
order_items → products              0
order_items → sellers               0
order_payments → orders             0
order_reviews → orders              0


### 결과

- 주요 Foreign Key 관계에서 참조 대상이 없는 레코드는 확인되지 않았다.
- 주요 Foreign Key 값이 참조 대상 부모 테이블의 Primary Key에 모두 존재하여 참조 무결성이 유지되었다.
- 따라서 주요 테이블 간 JOIN 및 KPI 계산에 영향을 줄 수 있는 참조 무결성 문제는 발견되지 않았다.

## 6. Numeric Range Check

주요 수치형 컬럼에 비정상적인 음수값 또는 허용 범위를 벗어난 값이 존재하는지 확인한다.

In [42]:
numeric_checks = {
    "order_items.price < 0": """
        SELECT COUNT(*) AS invalid_count
        FROM order_items
        WHERE price < 0;
    """,

    "order_items.freight_value < 0": """
        SELECT COUNT(*) AS invalid_count
        FROM order_items
        WHERE freight_value < 0;
    """,

    "order_payments.payment_value < 0": """
        SELECT COUNT(*) AS invalid_count
        FROM order_payments
        WHERE payment_value < 0;
    """,

    "order_payments.payment_installments < 1": """
        SELECT COUNT(*) AS invalid_count
        FROM order_payments
        WHERE payment_installments < 1;
    """,

    "order_reviews.review_score outside 1-5": """
        SELECT COUNT(*) AS invalid_count
        FROM order_reviews
        WHERE review_score < 1 OR review_score > 5;
    """,

    "products.product_weight_g < 0": """
        SELECT COUNT(*) AS invalid_count
        FROM products
        WHERE product_weight_g < 0;
    """,

    "products.product_length_cm < 0": """
        SELECT COUNT(*) AS invalid_count
        FROM products
        WHERE product_length_cm < 0;
    """,

    "products.product_height_cm < 0": """
        SELECT COUNT(*) AS invalid_count
        FROM products
        WHERE product_height_cm < 0;
    """,

    "products.product_width_cm < 0": """
        SELECT COUNT(*) AS invalid_count
        FROM products
        WHERE product_width_cm < 0;
    """
}

In [44]:
for check_name, query in numeric_checks.items():
    result = pd.read_sql(query, conn)
    invalid_count = result.loc[0, "invalid_count"]

    print(f"{check_name:<50} {invalid_count:,}")

order_items.price < 0                              0
order_items.freight_value < 0                      0
order_payments.payment_value < 0                   0
order_payments.payment_installments < 1            2
order_reviews.review_score outside 1-5             0
products.product_weight_g < 0                      0
products.product_length_cm < 0                     0
products.product_height_cm < 0                     0
products.product_width_cm < 0                      0


In [47]:
pd.read_sql("""
SELECT *
FROM order_payments
WHERE payment_installments < 1;
""", conn)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
1,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


### 결과

- 주요 금액(price, freight_value, payment_value)과 상품 크기 및 무게 컬럼에서는 음수값이 확인되지 않았다.
- 리뷰 점수는 모두 1~5점 범위 내에 존재하였다.
- `payment_installments`가 0인 레코드가 2건 확인되었다.
- 일반적인 할부 개수 기준에서는 검토가 필요한 값이지만, 전체 데이터에서 차지하는 비중이 매우 작아 분석에는 큰 영향을 주지 않는 것으로 판단하였다.

# Data Quality Summary

### Summary

- 모든 테이블이 정상적으로 적재되었다.
- 일부 컬럼에서 업무 특성에 따른 결측치가 확인되었다.
- Primary Key 기준 중복은 발견되지 않았으며, `order_reviews`는 `review_id + order_id` 조합으로 고유성이 확인되었다.
- 주문 데이터는 약 772일의 기간을 포함한다.
- 주요 Foreign Key 관계에서 참조 무결성 문제는 발견되지 않았다.
- 주요 수치형 컬럼은 정상 범위를 유지하였으며, `payment_installments = 0`인 레코드 2건만 추가 검토 대상으로 확인되었다.
- 전체적으로 Tableau Dashboard 구축 및 KPI 분석에 적합한 데이터 품질을 확인하였다.